# 8.4 Benchmarking — Apply

## Objective

Master inference benchmarking methodology for ONNX models. You will:

1. Build models of different sizes and benchmark them rigorously
2. Implement proper warmup and timing methodology
3. Compute percentile statistics (p50, p95, p99)
4. Run batch size sweeps and thread count comparisons
5. Calculate throughput in samples/second
6. Visualize latency distributions

**Key metric definitions:**

$$\text{Throughput} = \frac{B}{\bar{T}} \text{ samples/sec}$$

where $B$ is the batch size and $\bar{T}$ is the mean inference time.

$$\text{P}k = \text{value at the } k\text{-th percentile of latency distribution}$$

Production SLAs typically target **p99 latency** — the worst case for 99% of requests.

In [ ]:
# Setup
import numpy as np
import onnx
from onnx import helper, TensorProto, numpy_helper
from onnx.checker import check_model
import onnxruntime as ort
import time
import os
import tempfile
from collections import defaultdict

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

TMPDIR = tempfile.mkdtemp(prefix='onnx_bench_')
print(f"onnx {onnx.__version__}, onnxruntime {ort.__version__}")
print(f"Available providers: {ort.get_available_providers()}")

---
## Exercise 1: Build Benchmark Models of Different Sizes

We create three models (small, medium, large) to study how model complexity
affects inference performance. FLOPs for a linear layer:

$$\text{FLOPs}_{\text{MatMul}} = 2 \cdot M \cdot K \cdot N$$

(multiply-accumulate = 2 FLOPs per weight element)

In [ ]:
def build_mlp(dims, batch_dim='N', name='mlp'):
    """Build an MLP with given layer dimensions and dynamic batch."""
    np.random.seed(42)
    inits, nodes = [], []
    prev_name = 'X'

    for i in range(len(dims) - 1):
        d_in, d_out = dims[i], dims[i+1]
        W = np.random.randn(d_in, d_out).astype(np.float32) * np.sqrt(2.0/d_in)
        b = np.zeros(d_out, dtype=np.float32)
        inits.append(numpy_helper.from_array(W, f'W{i}'))
        inits.append(numpy_helper.from_array(b, f'b{i}'))
        nodes.append(helper.make_node('MatMul', [prev_name, f'W{i}'], [f'mm{i}']))
        nodes.append(helper.make_node('Add', [f'mm{i}', f'b{i}'], [f'a{i}']))
        if i < len(dims) - 2:
            nodes.append(helper.make_node('Relu', [f'a{i}'], [f'r{i}']))
            prev_name = f'r{i}'
        else:
            prev_name = f'a{i}'

    X_i = helper.make_tensor_value_info('X', TensorProto.FLOAT, [batch_dim, dims[0]])
    Y_i = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [batch_dim, dims[-1]])
    g = helper.make_graph(nodes, name, [X_i], [Y_i], initializer=inits)
    m = helper.make_model(g, opset_imports=[helper.make_opsetid('', 17)])
    check_model(m)
    return m

models = {
    'small':  build_mlp([64, 128, 64], name='small'),
    'medium': build_mlp([128, 256, 256, 128], name='medium'),
    'large':  build_mlp([256, 512, 512, 512, 256], name='large'),
}

# Compute FLOPs for each model (batch=1)
for mname, m in models.items():
    total_flops = 0
    total_params = 0
    for init in m.graph.initializer:
        arr = numpy_helper.to_array(init)
        total_params += arr.size
        if arr.ndim == 2:
            total_flops += 2 * arr.shape[0] * arr.shape[1]  # per sample
    path = os.path.join(TMPDIR, f'{mname}.onnx')
    onnx.save(m, path)
    print(f"{mname:<8}: {len(m.graph.node)} nodes, {total_params:>8,} params, "
          f"{total_flops:>10,} FLOPs/sample, {os.path.getsize(path)/1024:.1f} KB")

---
## Exercise 2: Warmup and Timing Methodology

Proper benchmarking requires:
1. **Warmup phase**: first few runs are slow (JIT compilation, cache population)
2. **Steady-state timing**: measure only after warmup
3. **Sufficient repetitions**: reduce variance from OS scheduling, power management

Use `time.perf_counter()` for wall-clock timing (includes I/O), not `time.process_time()`.

In [ ]:
def benchmark(sess, feeds, n_warmup=50, n_runs=1000):
    """Benchmark with proper warmup and per-iteration timing."""
    output_names = [o.name for o in sess.get_outputs()]

    # Warmup
    warmup_times = []
    for _ in range(n_warmup):
        t0 = time.perf_counter()
        sess.run(output_names, feeds)
        warmup_times.append((time.perf_counter() - t0) * 1000)

    # Timed runs
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        sess.run(output_names, feeds)
        times.append((time.perf_counter() - t0) * 1000)

    return np.array(warmup_times), np.array(times)

# Demonstrate warmup effect
sess = ort.InferenceSession(models['medium'].SerializeToString(),
                            providers=['CPUExecutionProvider'])
x = np.random.randn(32, 128).astype(np.float32)

warmup_times, steady_times = benchmark(sess, {'X': x}, n_warmup=30, n_runs=500)

print(f"Warmup phase ({len(warmup_times)} iterations):")
print(f"  First 5:  {warmup_times[:5].round(4)} ms")
print(f"  Last 5:   {warmup_times[-5:].round(4)} ms")
print(f"  Mean:     {warmup_times.mean():.4f} ms")
print(f"\nSteady-state ({len(steady_times)} iterations):")
print(f"  Mean:     {steady_times.mean():.4f} ms")
print(f"  Std:      {steady_times.std():.4f} ms")
print(f"  Min:      {steady_times.min():.4f} ms")
print(f"  Max:      {steady_times.max():.4f} ms")

warmup_overhead = warmup_times[:5].mean() / steady_times.mean()
print(f"\nFirst-run overhead: {warmup_overhead:.2f}x (vs steady state)")
print(f"This demonstrates why warmup is critical for accurate benchmarking.")

---
## Exercise 3: Percentile Statistics (p50, p95, p99)

For production systems, tail latency matters more than mean:

- **p50 (median):** "typical" request latency
- **p95:** 1 in 20 requests is slower than this
- **p99:** worst-case for 99% of traffic (SLA target)

The coefficient of variation $\text{CV} = \sigma / \mu$ measures timing stability.

In [ ]:
def compute_stats(times, batch_size=1):
    """Compute comprehensive benchmark statistics."""
    return {
        'n': len(times),
        'mean_ms': times.mean(),
        'std_ms': times.std(),
        'cv': times.std() / times.mean(),
        'min_ms': times.min(),
        'p50_ms': np.percentile(times, 50),
        'p95_ms': np.percentile(times, 95),
        'p99_ms': np.percentile(times, 99),
        'max_ms': times.max(),
        'throughput_sps': batch_size * 1000.0 / times.mean(),
    }

# Benchmark all three models
batch_size = 32
all_stats = {}

for mname, model in models.items():
    d_in = model.graph.input[0].type.tensor_type.shape.dim[1].dim_value
    sess = ort.InferenceSession(model.SerializeToString(),
                                providers=['CPUExecutionProvider'])
    x = np.random.randn(batch_size, d_in).astype(np.float32)
    _, times = benchmark(sess, {'X': x}, n_warmup=50, n_runs=2000)
    all_stats[mname] = compute_stats(times, batch_size)

print(f"Benchmark Results (batch={batch_size})")
print(f"{'Metric':<20} {'Small':>10} {'Medium':>10} {'Large':>10}")
print('=' * 52)
for metric in ['mean_ms', 'std_ms', 'cv', 'p50_ms', 'p95_ms', 'p99_ms',
               'max_ms', 'throughput_sps']:
    fmt = '.4f' if 'ms' in metric or metric == 'cv' else '.0f'
    label = metric.replace('_', ' ').replace('ms', '(ms)').replace('sps', '(samp/s)')
    vals = [f"{all_stats[m][metric]:{fmt}}" for m in ['small', 'medium', 'large']]
    print(f"{label:<20} {vals[0]:>10} {vals[1]:>10} {vals[2]:>10}")

# Tail latency amplification
for mname in all_stats:
    s = all_stats[mname]
    tail_amp = s['p99_ms'] / s['p50_ms']
    print(f"\n{mname}: p99/p50 tail amplification = {tail_amp:.2f}x")

---
## Exercise 4: Batch Size Sweep

Throughput typically increases with batch size (better hardware utilization)
until memory bandwidth or compute saturates:

$$\text{Throughput}(B) = \frac{B}{T(B)}, \quad
\frac{\partial T}{\partial B} \to 0 \text{ as } B \to \infty$$

Latency increases linearly at first, then sub-linearly due to amortization of
kernel launch overhead.

In [ ]:
batch_sizes = [1, 2, 4, 8, 16, 32, 64, 128, 256]
batch_results = []

model = models['medium']
d_in = 128

for bs in batch_sizes:
    sess = ort.InferenceSession(model.SerializeToString(),
                                providers=['CPUExecutionProvider'])
    x = np.random.randn(bs, d_in).astype(np.float32)
    _, times = benchmark(sess, {'X': x}, n_warmup=30, n_runs=1000)
    stats = compute_stats(times, bs)
    batch_results.append(stats)

print(f"Batch Size Sweep (medium model)")
print(f"{'Batch':>6} {'Mean(ms)':>10} {'P99(ms)':>10} {'Throughput':>12} {'Latency/samp':>14}")
print('-' * 55)
for bs, r in zip(batch_sizes, batch_results):
    per_sample = r['mean_ms'] / bs
    print(f"{bs:>6} {r['mean_ms']:>10.4f} {r['p99_ms']:>10.4f} "
          f"{r['throughput_sps']:>11.0f}/s {per_sample:>13.4f}ms")

if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    throughputs = [r['throughput_sps'] for r in batch_results]
    latencies = [r['mean_ms'] for r in batch_results]

    ax1.plot(batch_sizes, throughputs, 'o-', color='steelblue', linewidth=2, markersize=7)
    ax1.set_xlabel('Batch Size', fontsize=12)
    ax1.set_ylabel('Throughput (samples/sec)', fontsize=12)
    ax1.set_title('Throughput vs Batch Size', fontweight='bold')
    ax1.set_xscale('log', base=2)
    ax1.grid(alpha=0.3)

    ax2.plot(batch_sizes, latencies, 's-', color='coral', linewidth=2, markersize=7)
    ax2.set_xlabel('Batch Size', fontsize=12)
    ax2.set_ylabel('Total Latency (ms)', fontsize=12)
    ax2.set_title('Latency vs Batch Size', fontweight='bold')
    ax2.set_xscale('log', base=2)
    ax2.set_yscale('log')
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

---
## Exercise 5: Thread Count Comparison

ORT uses `intra_op_num_threads` for parallelism within operators (e.g., GEMM)
and `inter_op_num_threads` for executing independent operators in parallel.

Scaling efficiency:

$$\text{Efficiency}(n) = \frac{T_1}{n \cdot T_n} \times 100\%$$

Amdahl's Law limits the speedup from parallelism when serial portions exist.

In [ ]:
thread_counts = [1, 2, 4]
thread_results = []
batch_size = 32

for n_threads in thread_counts:
    so = ort.SessionOptions()
    so.intra_op_num_threads = n_threads
    so.inter_op_num_threads = 1
    sess = ort.InferenceSession(models['large'].SerializeToString(), so,
                                providers=['CPUExecutionProvider'])
    x = np.random.randn(batch_size, 256).astype(np.float32)
    _, times = benchmark(sess, {'X': x}, n_warmup=50, n_runs=1000)
    stats = compute_stats(times, batch_size)
    thread_results.append(stats)

baseline_time = thread_results[0]['mean_ms']
print(f"Thread Scaling (large model, batch={batch_size})")
print(f"{'Threads':>8} {'Mean(ms)':>10} {'P99(ms)':>10} {'Speedup':>8} {'Efficiency':>11}")
print('-' * 50)
for n, r in zip(thread_counts, thread_results):
    speedup = baseline_time / r['mean_ms']
    efficiency = speedup / n * 100
    print(f"{n:>8} {r['mean_ms']:>10.4f} {r['p99_ms']:>10.4f} {speedup:>7.2f}x {efficiency:>10.1f}%")

print(f"\nNote: Small models may not benefit from multiple threads due to")
print(f"thread synchronization overhead exceeding computation savings.")

---
## Exercise 6: Throughput Calculation and Analysis

Throughput measures processing capacity. For batch inference:

$$\text{Throughput}_{\text{total}} = \frac{\text{batch\_size}}{\text{mean\_latency\_sec}} \text{ samples/sec}$$

$$\text{Cost per inference} = \frac{\text{instance\_cost\_per\_hour}}{3600 \times \text{throughput}}$$

In [ ]:
# Comprehensive throughput analysis
configs = [
    ('small, bs=1',   'small',  1),
    ('small, bs=32',  'small',  32),
    ('small, bs=128', 'small',  128),
    ('medium, bs=1',  'medium', 1),
    ('medium, bs=32', 'medium', 32),
    ('medium, bs=128','medium', 128),
    ('large, bs=1',   'large',  1),
    ('large, bs=32',  'large',  32),
    ('large, bs=128', 'large',  128),
]

throughput_table = []

for label, mname, bs in configs:
    model = models[mname]
    d_in = model.graph.input[0].type.tensor_type.shape.dim[1].dim_value
    sess = ort.InferenceSession(model.SerializeToString(),
                                providers=['CPUExecutionProvider'])
    x = np.random.randn(bs, d_in).astype(np.float32)
    _, times = benchmark(sess, {'X': x}, n_warmup=30, n_runs=500)
    stats = compute_stats(times, bs)
    throughput_table.append((label, stats))

print(f"Throughput Analysis")
print(f"{'Config':<20} {'Latency(ms)':>12} {'Throughput':>14} {'Cost @$1/hr':>12}")
print('=' * 60)
for label, s in throughput_table:
    cost_per_inference = 1.0 / (3600 * s['throughput_sps']) * 100  # cents
    print(f"{label:<20} {s['mean_ms']:>12.4f} {s['throughput_sps']:>12.0f}/s "
          f"${cost_per_inference*10:.6f}/1k")

---
## Exercise 7: Latency Distribution Visualization

Latency distributions are rarely Gaussian — they typically show a right tail
from OS interrupts, garbage collection, and cache effects.

The **jitter** (p99 - p50) quantifies tail latency instability.

In [ ]:
# Collect detailed timing for visualization
sess = ort.InferenceSession(models['medium'].SerializeToString(),
                            providers=['CPUExecutionProvider'])
x = np.random.randn(32, 128).astype(np.float32)
_, detailed_times = benchmark(sess, {'X': x}, n_warmup=100, n_runs=5000)

stats = compute_stats(detailed_times, 32)

print(f"Detailed Statistics (5000 runs, medium model, batch=32):")
print(f"  Mean:  {stats['mean_ms']:.4f} ms")
print(f"  Std:   {stats['std_ms']:.4f} ms")
print(f"  CV:    {stats['cv']:.4f}")
print(f"  P50:   {stats['p50_ms']:.4f} ms")
print(f"  P95:   {stats['p95_ms']:.4f} ms")
print(f"  P99:   {stats['p99_ms']:.4f} ms")
print(f"  Jitter (p99-p50): {stats['p99_ms'] - stats['p50_ms']:.4f} ms")

if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Histogram
    axes[0].hist(detailed_times, bins=80, color='steelblue', edgecolor='black', alpha=0.8)
    for pval, color, label in [(50, 'green', 'P50'), (95, 'orange', 'P95'), (99, 'red', 'P99')]:
        v = np.percentile(detailed_times, pval)
        axes[0].axvline(v, color=color, linestyle='--', linewidth=2, label=f'{label}={v:.3f}ms')
    axes[0].set_xlabel('Latency (ms)')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Latency Distribution', fontweight='bold')
    axes[0].legend(fontsize=9)
    axes[0].grid(alpha=0.3)

    # Time series
    axes[1].plot(detailed_times, '.', markersize=1, alpha=0.5, color='steelblue')
    axes[1].axhline(stats['mean_ms'], color='red', linewidth=1, label=f'Mean={stats["mean_ms"]:.3f}ms')
    axes[1].set_xlabel('Iteration')
    axes[1].set_ylabel('Latency (ms)')
    axes[1].set_title('Latency Time Series', fontweight='bold')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    # CDF
    sorted_times = np.sort(detailed_times)
    cdf = np.arange(1, len(sorted_times) + 1) / len(sorted_times)
    axes[2].plot(sorted_times, cdf, color='steelblue', linewidth=2)
    for pval, color in [(50, 'green'), (95, 'orange'), (99, 'red')]:
        v = np.percentile(detailed_times, pval)
        axes[2].axhline(pval/100, color=color, linestyle=':', alpha=0.5)
        axes[2].axvline(v, color=color, linestyle='--', alpha=0.5)
    axes[2].set_xlabel('Latency (ms)')
    axes[2].set_ylabel('CDF')
    axes[2].set_title('Cumulative Distribution', fontweight='bold')
    axes[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

---
## Exercise 8: Optimization Level Impact on Latency

Measure how graph optimization affects latency for a model with fusible patterns.

In [ ]:
# Build a ConvNet with fusible patterns
np.random.seed(0)
C_in, C_out = 3, 32
inits = [
    numpy_helper.from_array(np.random.randn(C_out, C_in, 3, 3).astype(np.float32)*0.1, 'W'),
    numpy_helper.from_array(np.zeros(C_out, dtype=np.float32), 'b'),
    numpy_helper.from_array(np.ones(C_out, dtype=np.float32), 'sc'),
    numpy_helper.from_array(np.zeros(C_out, dtype=np.float32), 'bi'),
    numpy_helper.from_array(np.zeros(C_out, dtype=np.float32), 'mu'),
    numpy_helper.from_array(np.ones(C_out, dtype=np.float32), 'va'),
]
nodes = [
    helper.make_node('Conv', ['X', 'W', 'b'], ['c'], kernel_shape=[3,3], pads=[1,1,1,1]),
    helper.make_node('BatchNormalization', ['c', 'sc', 'bi', 'mu', 'va'], ['bn']),
    helper.make_node('Relu', ['bn'], ['Y']),
]
X_i = helper.make_tensor_value_info('X', TensorProto.FLOAT, [1, C_in, 64, 64])
Y_i = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [1, C_out, 64, 64])
g = helper.make_graph(nodes, 'conv_bn_relu', [X_i], [Y_i], initializer=inits)
cnn = helper.make_model(g, opset_imports=[helper.make_opsetid('', 17)])
check_model(cnn)

x_cnn = np.random.randn(1, C_in, 64, 64).astype(np.float32)
opt_levels = [
    ('DISABLED', ort.GraphOptimizationLevel.ORT_DISABLE_ALL),
    ('BASIC',    ort.GraphOptimizationLevel.ORT_ENABLE_BASIC),
    ('EXTENDED', ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED),
    ('ALL',      ort.GraphOptimizationLevel.ORT_ENABLE_ALL),
]

opt_bench = []
for name, level in opt_levels:
    so = ort.SessionOptions()
    so.graph_optimization_level = level
    sess = ort.InferenceSession(cnn.SerializeToString(), so,
                                providers=['CPUExecutionProvider'])
    _, times = benchmark(sess, {'X': x_cnn}, n_warmup=50, n_runs=1000)
    stats = compute_stats(times, 1)
    opt_bench.append((name, stats))

base_lat = opt_bench[0][1]['mean_ms']
print(f"Optimization Impact on Conv+BN+Relu (64x64 input):")
print(f"{'Level':<12} {'Mean(ms)':>10} {'P99(ms)':>10} {'Speedup':>8}")
print('-' * 42)
for name, s in opt_bench:
    print(f"{name:<12} {s['mean_ms']:>10.4f} {s['p99_ms']:>10.4f} {base_lat/s['mean_ms']:>7.2f}x")

---
## Challenge: Comprehensive Benchmark Suite

Build a reusable benchmark function that produces a complete report for any model:
model profile, multi-batch sweep, optimization comparison, and latency distribution.

In [ ]:
def full_benchmark_suite(model, input_shape_fn, batch_sizes=[1, 8, 32, 128],
                         n_warmup=50, n_runs=500):
    """Run a complete benchmark suite on an ONNX model.
    
    Args:
        model: ONNX ModelProto
        input_shape_fn: callable(batch_size) -> shape tuple
        batch_sizes: list of batch sizes to test
    """
    model_bytes = model.SerializeToString()
    input_name = model.graph.input[0].name

    # Model profile
    total_params = sum(numpy_helper.to_array(i).size for i in model.graph.initializer)
    model_size = len(model_bytes)
    print(f"{'='*60}")
    print(f"COMPREHENSIVE BENCHMARK REPORT")
    print(f"{'='*60}")
    print(f"Model: {model.graph.name}")
    print(f"  Nodes: {len(model.graph.node)}")
    print(f"  Parameters: {total_params:,}")
    print(f"  Size: {model_size/1024:.1f} KB")

    # Batch sweep
    print(f"\n--- Batch Size Sweep ---")
    print(f"{'Batch':>6} {'Mean(ms)':>10} {'P50(ms)':>10} {'P99(ms)':>10} {'Throughput':>12}")
    print('-' * 50)
    sweep_data = []
    for bs in batch_sizes:
        shape = input_shape_fn(bs)
        sess = ort.InferenceSession(model_bytes, providers=['CPUExecutionProvider'])
        x = np.random.randn(*shape).astype(np.float32)
        _, times = benchmark(sess, {input_name: x}, n_warmup, n_runs)
        s = compute_stats(times, bs)
        sweep_data.append((bs, s))
        print(f"{bs:>6} {s['mean_ms']:>10.4f} {s['p50_ms']:>10.4f} "
              f"{s['p99_ms']:>10.4f} {s['throughput_sps']:>10.0f}/s")

    # Best throughput
    best_bs, best_s = max(sweep_data, key=lambda x: x[1]['throughput_sps'])
    print(f"\n  Best throughput: {best_s['throughput_sps']:.0f} samp/s at batch={best_bs}")

    # Optimization comparison (at best batch size)
    print(f"\n--- Optimization Levels (batch={best_bs}) ---")
    shape = input_shape_fn(best_bs)
    x = np.random.randn(*shape).astype(np.float32)
    levels = [
        ('DISABLED', ort.GraphOptimizationLevel.ORT_DISABLE_ALL),
        ('ALL',      ort.GraphOptimizationLevel.ORT_ENABLE_ALL),
    ]
    for lname, level in levels:
        so = ort.SessionOptions()
        so.graph_optimization_level = level
        sess = ort.InferenceSession(model_bytes, so, providers=['CPUExecutionProvider'])
        _, times = benchmark(sess, {input_name: x}, n_warmup, n_runs)
        s = compute_stats(times, best_bs)
        print(f"  {lname:<10}: mean={s['mean_ms']:.4f}ms, p99={s['p99_ms']:.4f}ms, "
              f"throughput={s['throughput_sps']:.0f}/s")

    print(f"\n{'='*60}")
    return sweep_data

# Run the suite on our medium model
result = full_benchmark_suite(
    models['large'],
    input_shape_fn=lambda bs: (bs, 256),
    batch_sizes=[1, 4, 16, 64, 128, 256],
)

---
## Summary

| Concept | What You Practiced |
|:---|:---|
| Warmup | Demonstrated first-run overhead, proper warmup methodology |
| Percentiles | Computed p50, p95, p99; measured tail latency amplification |
| Batch sweep | Throughput vs batch size curve, per-sample amortization |
| Thread scaling | `intra_op_num_threads` effect, scaling efficiency |
| Throughput | $B \cdot 1000 / \bar{T}$ samples/sec, cost analysis |
| Distribution | Histogram, CDF, time-series visualization |
| Optimization impact | DISABLED vs ALL latency comparison |

**Key formulas:**

$$\text{Throughput} = \frac{B}{\bar{T}}, \quad
\text{Efficiency}(n) = \frac{T_1}{n T_n}, \quad
\text{CV} = \frac{\sigma}{\mu}$$

**Next:** [Inspecting Models](../../09_ONNX_Graph_Manipulation/01_Inspecting_Models/)